In [1]:
import ollama
import numpy as np
from pymilvus import connections, Collection
import time
from rank_bm25 import BM25Okapi

In [2]:
connections.connect(
    alias="default",
    host="localhost",
    port="19530"
)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\3293735357.py:1: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


In [4]:
index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "COSINE",
    "params": {
        "nlist": 128
    }
}

In [3]:
collection = Collection(
    name="nutrition_rag"
)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\2368230308.py:1: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(


In [6]:
collection.create_index(
    field_name="embedding",
    index_params=index_params
)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_23712\3225226010.py:1: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(


Status(code=0, message=)

In [4]:
def convert_embeddings(prompt):
    response = ollama.embeddings(model="nomic-embed-text:v1.5", prompt=prompt)
    return np.array(response['embedding'])

In [ ]:
query = "how many dietary guidelines are available? how will this help me with my diet?"
query_embedding = convert_embeddings(query)

search_params = {
    "metric_type": "COSINE",
    "params": {
        "nprobe": 10
    }
}

In [31]:
start = time.time()

results = collection.search(
    data=[query_embedding],
    anns_field="embedding",
    param=search_params,
    limit=5,
    expr='chunk_type == "recursive_character"',
    output_fields=[
        "text",
        "chunk_type"
    ]
)

end = time.time()

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_9232\1998789969.py:3: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


In [33]:
dense_retrival_text = []
for hits in results:

    for i, hit in enumerate(hits):

        print(f"\nRESULT #{i+1}")
        print("-" * 50)

        print("Similarity Score:")
        print(hit.score)

        print("\nChunk Type:")
        print(hit.entity.get("chunk_type"))

        print("\nRetrieved Text:")
        print(hit.entity.get("text"))
        
        dense_retrival_text.append(hit.entity.get("text"))



RESULT #1
--------------------------------------------------
Similarity Score:
0.671568751335144

Chunk Type:
recursive_character

Retrieved Text:
also agreed to halt the rise in diabetes and obesity in adults and adolescents as well as in childhood over-
weight by 2025 (9, 10).
OVERVIEW
Consuming a healthy diet throughout the life-course helps to prevent malnutrition in all its forms as well as a 
range of noncommunicable diseases (NCDs) and conditions. However, increased production of processed foods, 
rapid urbanization and changing lifestyles have led to a shift in dietary patterns. People are now consuming 
more foods high in energy, fats, free sugars and salt/sodium, and many people do not eat enough fruit, vegeta-
bles and other dietary fibre such as whole grains.
The exact make-up of a diversified, balanced and healthy diet will vary depending on individual characteristics 
(e.g. age, gender, lifestyle and degree of physical activity), cultural context, locally available foods

## BM25 retrival

In [35]:
query_results = collection.query(
    expr="id >= 0",
    output_fields=["text"]
)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_9232\2470714968.py:1: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  query_results = collection.query(


In [36]:
"""
BM25 works ONLY on raw text.

It does NOT use embeddings.
"""

all_chunks = [r["text"] for r in query_results]

print("Total chunks:", len(all_chunks))

Total chunks: 163


In [37]:
tokenized_chunks = [
    chunk.lower().split()
    for chunk in all_chunks
]

In [ ]:
query = "how many dietary guidelines are available? how will this help me with my diet?"

tokenized_query = query.lower().split()

In [ ]:
bm25 = BM25Okapi(tokenized_query)

top_chunks = bm25.get_top_n(
    tokenized_query,
    all_chunks,
    n=5
)

In [42]:
for i, chunk in enumerate(top_chunks):

    print(f"\nRESULT {i+1}")
    print("-" * 50)

    print(chunk)


RESULT 1
--------------------------------------------------
HOW TO PROMOTE HEALTHY DIETS 
Diet evolves over time, being influenced by many social and economic factors that interact in a complex manner 
to shape individual dietary patterns.

RESULT 2
--------------------------------------------------
Some food manufacturers are reformulating recipes to reduce the sodium content of their products, and people 
should be encouraged to check nutrition labels to see how much sodium is in a product before purchasing or 
consuming it.

RESULT 3
--------------------------------------------------
People are now consuming 
more foods high in energy, fats, free sugars and salt/sodium, and many people do not eat enough fruit, vegeta-
bles and other dietary fibre such as whole grains.

RESULT 4
--------------------------------------------------
candies and sugar-sweetened beverages (i.e. all types of beverages containing free sugars – these include car-
bonated or non-carbonated soft drinks, fruit 

In [48]:
combined = dense_retrival_text + top_chunks

In [50]:
unique_results = list(
        dict.fromkeys(combined)
    )

In [5]:
def dense_retrieval(query, top_k=5):

    query_embedding = ollama.embeddings(
        model="nomic-embed-text:v1.5",
        prompt=query
    )["embedding"]

    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={
            "metric_type": "COSINE",
            "params": {"nprobe": 10}
        },
        limit=top_k,
        output_fields=["text"]
    )

    dense_results = []

    for hits in results:
        for hit in hits:
            dense_results.append({
                "text": hit.entity.get("text"),
                "score": float(hit.score),
                "source": "dense"
            })

    return dense_results

In [32]:
query = "pesticide  residues  from  the  food  products"
dense_retrieval(query)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\3783350138.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


[{'text': 'Therefore, promoting a healthy food environment – including \nfood systems that promote a diversified, balanced and healthy diet – requires the involvement of multiple sectors \nand stakeholders, including government, and the public and private sectors. Governments have a central role in creating a healthy food environment that enables people to adopt and main-\ntain healthy dietary practices. Effective actions by policy-makers to create a healthy food environment include \nthe following:\nn Creating coherence in national policies and investment plans – including trade, food and agricultural policies \n– to promote a healthy diet and protect public health through:\n—\n— increasing incentives for producers and retailers to grow, use and sell fresh fruit and vegetables;\n—\n— reducing incentives for the food industry to continue or increase production of processed foods contain-\ning high levels of saturated fats, trans-fats, free sugars and salt/sodium;\n—\n— encouraging refo

In [ ]:
results = collection.query(
    expr="id >= 0",
    output_fields=["text"]
)

all_chunks = [
    r["text"]
    for r in results
]

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\3303169299.py:1: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.query(


In [24]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

def get_all_chunks():
    results = collection.query(
        expr="id >= 0",
        output_fields=["text"]
    )

    all_chunks = [
        r["text"]
        for r in results
    ]
    
    return all_chunks


# =========================================
# CREATE BM25 INDEX (ONLY ONCE)
# =========================================

all_chunks = get_all_chunks()

cleaned_chunks = [
    clean_text(chunk)
    for chunk in all_chunks
]

tokenized_chunks = [
    chunk.split()
    for chunk in cleaned_chunks
]

bm25 = BM25Okapi(tokenized_chunks)

# =========================================
# BM25 RETRIEVAL
# =========================================

def bm25_retrieval(query, top_k=5):

    cleaned_query = clean_text(query)
    tokenized_query = cleaned_query.split()
    scores = bm25.get_scores(tokenized_query)

    chunk_score_pairs = list(
        zip(all_chunks, scores)
    )

    ranked_results = sorted(
        chunk_score_pairs,
        key=lambda x: x[1],
        reverse=True
    )

    top_results = ranked_results[:top_k]
    
    formatted_results = []
    for chunk, score in top_results:
        formatted_results.append({
            "text": chunk,
            "score": float(score),
            "source": "bm25"
        })

    return formatted_results

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\3755264554.py:9: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.query(


In [25]:
query = "pesticide  residues  from  the  food  products"
bm25_retrieval(query, top_k=10)

[{'text': 'Some food manufacturers are reformulating recipes to reduce the sodium content of their products, and people \nshould be encouraged to check nutrition labels to see how much sodium is in a product before purchasing or \nconsuming it.',
  'score': 5.552600065891788,
  'source': 'bm25'},
 {'text': 'cessed foods (e.g. ready meals; processed meats such as bacon, ham and salami; cheese; and salty snacks) or from \nfoods consumed frequently in large amounts (e.g. bread). Salt is also added to foods during cooking (e.g. bouil-\nlon, stock cubes, soy sauce and fish sauce) or at the point of consumption (e.g. table salt).\nSalt intake can be reduced by:\nn limiting the amount of salt and high-sodium condiments (e.g. soy sauce, fish sauce and bouillon) when cook-\ning and preparing foods;\nn not having salt or high-sodium sauces on the table;\nn limiting the consumption of salty snacks; and\nn choosing products with lower sodium content.\nSome food manufacturers are reformulating reci

In [26]:
def hybrid_retrieval(query, dense_k=5, bm25_k=5):

    dense_results = dense_retrieval(
        query,
        top_k=dense_k
    )

    bm25_results = bm25_retrieval(
        query,
        top_k=bm25_k
    )

    combined = dense_results + bm25_results

    unique_results = list(
        {
            item["text"]: item
            for item in combined
        }.values()
    )

    return unique_results

In [27]:
hybrid_results = hybrid_retrieval(query, dense_k=5, bm25_k=5)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\3783350138.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


In [29]:
hybrid_results

[{'text': 'Therefore, promoting a healthy food environment – including \nfood systems that promote a diversified, balanced and healthy diet – requires the involvement of multiple sectors \nand stakeholders, including government, and the public and private sectors. Governments have a central role in creating a healthy food environment that enables people to adopt and main-\ntain healthy dietary practices. Effective actions by policy-makers to create a healthy food environment include \nthe following:\nn Creating coherence in national policies and investment plans – including trade, food and agricultural policies \n– to promote a healthy diet and protect public health through:\n—\n— increasing incentives for producers and retailers to grow, use and sell fresh fruit and vegetables;\n—\n— reducing incentives for the food industry to continue or increase production of processed foods contain-\ning high levels of saturated fats, trans-fats, free sugars and salt/sodium;\n—\n— encouraging refo

In [30]:
def weighted_hybrid_retrieval(query,
                              dense_weight=0.7,
                              bm25_weight=0.3,
                              top_k=5):

    dense_results = dense_retrieval(query, top_k=top_k)
    bm25_results = bm25_retrieval(query, top_k=top_k)
    combined = {}

    # DENSE RESULTS
    for r in dense_results:
        text = r["text"]
        combined[text] = {
            "text": text,
            "score": r["score"] * dense_weight
        }

    # BM25 RESULTS
    for r in bm25_results:
        text = r["text"]
        if text in combined:
            combined[text]["score"] += (
                r["score"] * bm25_weight
            )
        else:
            combined[text] = {
                "text": text,
                "score": r["score"] * bm25_weight
            }

    ranked = sorted(
        combined.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    return ranked[:top_k]

In [52]:
weigh_hybrid_retrival_results = weighted_hybrid_retrieval(query, dense_weight=0.7, bm25_weight=0.3, top_k=5)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\3783350138.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


In [33]:
def generate_hypothetical_answer(query):

    response = ollama.chat(
        model="qwen3.5:cloud",

        messages=[
            {
                "role": "user",
                "content": f"Answer briefly: {query}"
            }
        ]
    )

    return response["message"]["content"]

In [36]:
def hyde_retrieval(query, top_k=5):

    hypothetical_answer = (
        generate_hypothetical_answer(query)
    )

    query_embedding = convert_embeddings(hypothetical_answer)

    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={
            "metric_type": "COSINE",
            "params": {
                "nprobe": 10
            }
        },
        limit=top_k,
        output_fields=["text"]
    )

    formatted_results = []
    for hits in results:
        for hit in hits:
            formatted_results.append({
                "text": hit.entity.get("text"),
                "score": float(hit.score),
                "source": "hyde"
            })

    return formatted_results

In [37]:
hyde_results = hyde_retrieval(query, top_k=5)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_7980\675363175.py:9: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


In [38]:
hyde_results

[{'text': 'Therefore, promoting a healthy food environment – including \nfood systems that promote a diversified, balanced and healthy diet – requires the involvement of multiple sectors \nand stakeholders, including government, and the public and private sectors. Governments have a central role in creating a healthy food environment that enables people to adopt and main-\ntain healthy dietary practices. Effective actions by policy-makers to create a healthy food environment include \nthe following:\nn Creating coherence in national policies and investment plans – including trade, food and agricultural policies \n– to promote a healthy diet and protect public health through:\n—\n— increasing incentives for producers and retailers to grow, use and sell fresh fruit and vegetables;\n—\n— reducing incentives for the food industry to continue or increase production of processed foods contain-\ning high levels of saturated fats, trans-fats, free sugars and salt/sodium;\n—\n— encouraging refo

### Re-Ranking Approach

In [39]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

c:\Users\Aathi K M\Documents\ai_portfolio_projects\simple_rag_pipeline_development\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Aathi K M\Documents\ai_portfolio_projects\simple_rag_pipeline_development\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aathi K M\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Wind

In [40]:
def rerank(query, retrieved_chunks):

    pairs = [
        [query, chunk["text"]]
        for chunk in retrieved_chunks
    ]
    scores = reranker.predict(pairs)

    reranked = []
    for chunk, score in zip(retrieved_chunks, scores):
        reranked.append({
            "text": chunk["text"],
            "score": float(score)
        })

    reranked = sorted(
        reranked,
        key=lambda x: x["score"],
        reverse=True
    )

    return reranked

In [41]:
reranked_results = rerank(query, hyde_results)

In [42]:
reranked_results

[{'text': 'cessed foods (e.g. ready meals; processed meats such as bacon, ham and salami; cheese; and salty snacks) or from \nfoods consumed frequently in large amounts (e.g. bread). Salt is also added to foods during cooking (e.g. bouil-\nlon, stock cubes, soy sauce and fish sauce) or at the point of consumption (e.g. table salt).\nSalt intake can be reduced by:\nn limiting the amount of salt and high-sodium condiments (e.g. soy sauce, fish sauce and bouillon) when cook-\ning and preparing foods;\nn not having salt or high-sodium sauces on the table;\nn limiting the consumption of salty snacks; and\nn choosing products with lower sodium content.\nSome food manufacturers are reformulating recipes to reduce the sodium content of their products, and people \nshould be encouraged to check nutrition labels to see how much sodium is in a product before purchasing or \nconsuming it.\nPotassium can mitigate the negative effects of elevated sodium consumption on blood pressure. Intake of potas

### LLM Retrival part

In [43]:
def build_context(results):

    context = ""
    for i, result in enumerate(results):
        chunk = result["text"]
        context += f"\n\nChunk {i+1}:\n"
        context += chunk
        
    return context

In [56]:
## RAG prompt
def create_rag_prompt(query, context):

    
        # Answer the user's question ONLY
        # using the provided context.

        # If the answer is not available
        # in the context, say:

        # 'I could not find relevant information.'
    prompt = f"""
        You are a nutrition assistant.
        
        Answer the question using the context.

        ----------------------------
        Context:
        {context}
        
        ----------------------------
        Question:
        {query}

        Answer:
    """

    return prompt

In [57]:
def generate_response(prompt):

    response = ollama.chat(
        model="kimi-k2.5:cloud",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [58]:
def rag_pipeline(retrieved_chunks, query):

    # BUILD CONTEXT
    context = build_context(
        retrieved_chunks
    )

    # CREATE PROMPT
    prompt = create_rag_prompt(
        query,
        context
    )

    # GENERATE RESPONSE
    answer = generate_response(prompt)

    return answer

In [59]:
rag_pipeline(retrieved_chunks=weigh_hybrid_retrival_results, query=query)

'The provided context does not contain information regarding pesticide residues from food products. \n\nThe context focuses on:\n- Reducing sodium/salt intake and reformulating food products to lower sodium content\n- Eliminating industrially-produced trans-fats\n- Creating healthy food environments through nutrition labeling and marketing regulations\n- Encouraging consumption of fresh fruits and vegetables and limiting processed foods\n\nFor information on pesticide residues, you would need to consult resources specifically addressing food safety, agricultural chemical use, or maximum residue limits (MRLs) established by food safety authorities.'

In [48]:
build_context(hyde_results)

'\n\nChunk 1:\nTherefore, promoting a healthy food environment – including \nfood systems that promote a diversified, balanced and healthy diet – requires the involvement of multiple sectors \nand stakeholders, including government, and the public and private sectors. Governments have a central role in creating a healthy food environment that enables people to adopt and main-\ntain healthy dietary practices. Effective actions by policy-makers to create a healthy food environment include \nthe following:\nn Creating coherence in national policies and investment plans – including trade, food and agricultural policies \n– to promote a healthy diet and protect public health through:\n—\n— increasing incentives for producers and retailers to grow, use and sell fresh fruit and vegetables;\n—\n— reducing incentives for the food industry to continue or increase production of processed foods contain-\ning high levels of saturated fats, trans-fats, free sugars and salt/sodium;\n—\n— encouraging 

In [54]:
query

'pesticide  residues  from  the  food  products'

In [55]:
weigh_hybrid_retrival_results

[{'text': 'Some food manufacturers are reformulating recipes to reduce the sodium content of their products, and people \nshould be encouraged to check nutrition labels to see how much sodium is in a product before purchasing or \nconsuming it.',
  'score': 1.6657800197675363},
 {'text': 'Therefore, promoting a healthy food environment – including \nfood systems that promote a diversified, balanced and healthy diet – requires the involvement of multiple sectors \nand stakeholders, including government, and the public and private sectors. Governments have a central role in creating a healthy food environment that enables people to adopt and main-\ntain healthy dietary practices. Effective actions by policy-makers to create a healthy food environment include \nthe following:\nn Creating coherence in national policies and investment plans – including trade, food and agricultural policies \n– to promote a healthy diet and protect public health through:\n—\n— increasing incentives for produ